# Qwen3B: evaluation and comparison only

Compare a training-majority classifier, untouched Qwen2.5-Coder-3B-Instruct, and the same base with your trained adapter. No training or dataset download cells.

Use cloud NVIDIA for inference; use the Mac to read cached results. Select the project `.venv` kernel. Copy the trained adapter, its tokenizer, and `run_manifest.json` from a completed run. This notebook requires the frozen Stage 2 split manifest embedded in that run; it deliberately rejects historical checkpoints with unknown training membership.

Default: frozen **validation generation sample**. Final test requires explicitly freezing all choices. The adapter must have been trained under the supplied manifest. The code checks metadata consistency, not the truth of manually edited metadata.


In [ ]:
from pathlib import Path
import sys, json
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/evaluation/comparison.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation.comparison import prepare_data, QwenComparison, evaluate_cached, summarize
from src.evaluation.splits import digest

ADAPTER_DIR = PROJECT_ROOT / "checkpoints" / "SET_COMPLETED_ADAPTER_DIRECTORY"
RUN_MANIFEST = ADAPTER_DIR / "run_manifest.json"
EXTRACT_ROOT = PROJECT_ROOT / "data/contextcrbench/extracted"
SPLIT_MANIFEST = PROJECT_ROOT / "data/contextcrbench/splits/repository_80_10_10_v1.json"
SPLIT = "validation"  # or "test", only after protocol freeze
FINAL_TEST_PROTOCOL_FROZEN = False
RUN_INFERENCE = False  # Explicit opt-in; cached analysis below works without a GPU.
BASE_REVISION = "main"  # Prefer the exact training revision; resolved commit is saved.
OUTPUT_DIR = PROJECT_ROOT / "results/midpoint/qwen3b_validation_v1"


In [ ]:
if RUN_INFERENCE:
    manifest, run, examples, constant = prepare_data(
        EXTRACT_ROOT, SPLIT_MANIFEST, RUN_MANIFEST,
        split=SPLIT, final_test=FINAL_TEST_PROTOCOL_FROZEN)
    print(f"Split: {SPLIT}; examples: {len(examples)}; training-majority label: {constant}")
    print("Demo IDs:", manifest["demo_ids"])
    print("Verify base revision against the completed run before interpreting results.")
else:
    print("Inference disabled. Set paths and opt in on NVIDIA, or load existing results below.")


In [ ]:
if RUN_INFERENCE:
    evaluator = QwenComparison(ADAPTER_DIR, max_seq_length=run["max_seq_length"], revision=run.get("base_revision", BASE_REVISION))
    protocol = {**evaluator.identity, "split": SPLIT,
                "split_manifest_hash": digest(manifest), "training_run_hash": digest(run),
                "demo_ids": manifest["demo_ids"] if SPLIT == "validation" else [],
                "scoring": "strict JSON; invalid always incorrect; macro-F1 over False and True"}
    rows = evaluate_cached(examples, constant, evaluator.predict, protocol, OUTPUT_DIR)
    print(f"Saved {len(rows)} prediction records to {OUTPUT_DIR}")


## Cached results — no model loading

All three methods must have identical example IDs. Accuracy includes malformed outputs as failures; macro-F1 uses the two real classes, with invalid predictions treated as abstentions (false negatives for their true class). Majority has no generation cost. Latency includes generation/synchronization but excludes model loading and prompt tokenization; it is descriptive, not a controlled throughput benchmark.


In [ ]:
import pandas as pd
if not (OUTPUT_DIR / "predictions.jsonl").exists():
    print("No completed comparison yet. Run inference on the cloud and copy its results directory here.")
    predictions = None
else:
    rows = [json.loads(line) for line in (OUTPUT_DIR / "predictions.jsonl").read_text().splitlines()]
    protocol = json.loads((OUTPUT_DIR / "protocol.json").read_text())
    predictions = pd.DataFrame(rows)
    for name in ["majority", "base", "adapter"]:
        part = predictions[predictions.model == name]
        assert len(part) == len(protocol["ids"]) and set(part.id) == set(protocol["ids"])
        assert all(part.protocol_hash == digest(protocol))
    display(pd.DataFrame(summarize(rows)).set_index("model"))
    print("Evaluation partition:", protocol["split"])


In [ ]:
if predictions is not None:
    import matplotlib.pyplot as plt
    metrics = pd.DataFrame(summarize(rows)).set_index("model")
    ax = metrics[["accuracy", "macro_f1"]].plot.bar(ylim=(0, 1), rot=0, figsize=(7, 4))
    ax.set_title(f"Qwen3B comparison — {protocol['split']} (N={len(protocol['ids'])})")
    ax.set_ylabel("Score")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
    plt.show()
    paired = predictions[predictions.model.isin(["base", "adapter"])].pivot(index="id", columns="model", values="correct")
    display(paired.groupby(["base", "adapter"]).size().rename("examples").reset_index())


In [ ]:
if predictions is not None:
    # Fixed examples chosen before inference, not selected for favorable results.
    demo = predictions[predictions.id.isin(protocol["demo_ids"])]
    display(demo[["id", "model", "label", "raw_output", "valid", "correct"]])


## Interpretation and next steps

Report sample size, partition, class support, malformed rate, and both gains and regressions. These plots do not establish general coding-agent ability or the main learned-selector contribution. Confidence intervals, confusion matrices, and deeper error analysis can be added from `predictions.jsonl` without rerunning models.

Cache records are written per example/model and resumed only under the identical protocol. A changed model, adapter, prompt, dataset, tokenizer, or generation setting requires a new output directory. Do not repeatedly tune against final-test results. This workflow compares the same inference quantization for both models; it does not claim identity with the training numeric configuration.
